## Linear Classifier in TensorFlow 
Using Low Level API in Eager Execution mode

### Load tensorflow

In [0]:
import tensorflow as tf

In [0]:
#Enable Eager Execution if using tensflow version < 2.0
#From tensorflow v2.0 onwards, Eager Execution will be enabled by default


### Collect Data

In [0]:
from google.colab import drive
drive.mount('/content/drive/')

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).


In [0]:
import pandas as pd
import numpy as np

In [0]:
df = pd.read_csv('/content/drive/My Drive/Colab Notebooks/prices.csv')

### Check all columns in the dataset

In [101]:
df.head(5)

,date,symbol,open,close,low,high,volume
0,2016-01-05 00:00:00,WLTW,123.430000,125.839996,122.309998,126.250000,2163600.0
1,2016-01-06 00:00:00,WLTW,125.239998,119.980003,119.940002,125.540001,2386400.0
2,2016-01-07 00:00:00,WLTW,116.379997,114.949997,114.930000,119.739998,2489500.0
3,2016-01-08 00:00:00,WLTW,115.480003,116.620003,113.500000,117.440002,2006300.0
4,2016-01-11 00:00:00,WLTW,117.010002,114.970001,114.089996,117.330002,1408600.0


### Drop columns `date` and  `symbol`

In [102]:
df = df.drop(["date","symbol"],axis=1)
df.head(5)

,open,close,low,high,volume
0,123.430000,125.839996,122.309998,126.250000,2163600.0
1,125.239998,119.980003,119.940002,125.540001,2386400.0
2,116.379997,114.949997,114.930000,119.739998,2489500.0
3,115.480003,116.620003,113.500000,117.440002,2006300.0
4,117.010002,114.970001,114.089996,117.330002,1408600.0


In [103]:
df.columns

Index([u'open', u'close', u'low', u'high', u'volume'], dtype='object')

volume is the target variable

### Consider only first 1000 rows in the dataset for building feature set and target set
Target 'Volume' has very high values. Divide 'Volume' by 1000,000

In [104]:
df["volume"]=df["volume"]/1000000
df.head(5)

,open,close,low,high,volume
0,123.430000,125.839996,122.309998,126.250000,2.1636
1,125.239998,119.980003,119.940002,125.540001,2.3864
2,116.379997,114.949997,114.930000,119.739998,2.4895
3,115.480003,116.620003,113.500000,117.440002,2.0063
4,117.010002,114.970001,114.089996,117.330002,1.4086


### Divide the data into train and test sets

In [105]:
from sklearn.model_selection import train_test_split
X = df.drop("volume",axis=1)
y = df["volume"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.3, random_state=5)
print("Dataset Size: X_train: {}, X_test: {}, y_train {}, y_test: {}".format(X_train.shape, X_test.shape, y_train.shape, y_test.shape))

Dataset Size: X_train: (595884, 4), X_test: (255380, 4), y_train (595884,), y_test: (255380,)


#### Convert Training and Test Data to numpy float32 arrays


In [0]:
X_train = np.float32(X_train)
X_test = np.float32(X_test)
y_train = np.float32(y_train)
y_test = np.float32(y_test)

### Normalize the data
You can use Normalizer from sklearn.preprocessing

In [0]:
#Normalize the data

#from sklearn.preprocessing import Normalizer
#X_train = Normalizer().fit(X_train)
#X_test = Normalizer().fit(X_test)

from sklearn.preprocessing import normalize
#x_n = tf.nn.l2_normalize(x,1)
x_train_n = normalize(np.vstack([X_train]), norm='l2', axis=1)
x_test_n = normalize(np.vstack([X_test]), norm='l2', axis=1)

In [108]:
x_train_n

array([[0.5022512 , 0.4969147 , 0.49592185, 0.5048574 ],
       [0.49722308, 0.50261015, 0.49506828, 0.5050343 ],
       [0.50180554, 0.498122  , 0.49792814, 0.50212866],
       ...,
       [0.49939367, 0.5003606 , 0.4981851 , 0.5020526 ],
       [0.49761617, 0.50228006, 0.49563992, 0.50441444],
       [0.5004052 , 0.49883908, 0.49734756, 0.5033882 ]], dtype=float32)

In [109]:
x_test_n

array([[0.49931753, 0.50040627, 0.49822882, 0.5020394 ],
       [0.4971    , 0.5046832 , 0.4884071 , 0.50955373],
       [0.4979417 , 0.50077313, 0.49618164, 0.5050584 ],
       ...,
       [0.5031815 , 0.49751222, 0.496083  , 0.5031815 ],
       [0.49896017, 0.5009022 , 0.49896017, 0.50117314],
       [0.4996356 , 0.5018904 , 0.49625346, 0.50219786]], dtype=float32)

## Building the Model in tensorflow

1.Define Weights and Bias, use tf.zeros to initialize weights and Bias

In [0]:
W = tf.Variable(tf.zeros(shape=[4,1]), name="Weights")
b = tf.Variable(tf.zeros(shape=[1]),name="Bias")

2.Define a function to calculate prediction

In [0]:
yPred = tf.add(tf.matmul(x_train_n,W),b,name='output')

In [115]:
yPred.get_shape()

TensorShape([Dimension(595884), Dimension(1)])

3.Loss (Cost) Function [Mean square error]

In [0]:
loss = tf.reduce_mean(tf.square(y_train-yPred),name='Loss')

In [119]:
loss

<tf.Tensor 'Loss_1:0' shape=() dtype=float32>

4.Function to train the Model

1.   Record all the mathematical steps to calculate Loss
2.   Calculate Gradients of Loss w.r.t weights and bias
3.   Update Weights and Bias based on gradients and learning rate to minimize loss

In [0]:
train_op = tf.train.GradientDescentOptimizer(0.03).minimize(loss)

In [121]:
train_op

<tf.Operation 'GradientDescent' type=NoOp>

## Train the model for 100 epochs 
1. Observe the training loss at every iteration
2. Observe Train loss at every 5th iteration

In [127]:
#Lets start graph Execution
sess = tf.Session()

# variables need to be initialized before we can use them
sess.run(tf.global_variables_initializer())

y_pred , train_loss = sess.run([train_op, loss], feed_dict={x_train_n:x_train_n, y:y_train})

TypeError: ignored

In [0]:


#how many times data need to be shown to model
training_epochs = 100

for epoch in range(training_epochs):
            
    #Calculate train_op and loss
    _, train_loss = sess.run([train_op,loss],feed_dict={x:features, y_:actual_prices})
    
    if epoch % 5 == 0:
        print ('Training loss at step: ', epoch, ' is ', train_loss)

### Get the shapes and values of W and b

### Model Prediction on 1st Examples in Test Dataset

## Classification using tf.Keras

In this exercise, we will build a Deep Neural Network using tf.Keras. We will use Iris Dataset for this exercise.

### Load the given Iris data using pandas (Iris.csv)

### Target set has different categories. So, Label encode them. And convert into one-hot vectors using get_dummies in pandas.

### Splitting the data into feature set and target set

###  Building Model in tf.keras

Build a Linear Classifier model  <br>
1.  Use Dense Layer  with input shape of 4 (according to the feature set) and number of outputs set to 3<br> 
2. Apply Softmax on Dense Layer outputs <br>
3. Use SGD as Optimizer
4. Use categorical_crossentropy as loss function 

### Model Training 

### Model Prediction

### Save the Model

### Build and Train a Deep Neural network with 2 hidden layer  - Optional - For Practice

Does it perform better than Linear Classifier? What could be the reason for difference in performance?